In [1]:
import os
import pickle
import sqlite3

import numpy as np
import pandas as pd
import joblib
import faiss
import shap

from sentence_transformers import SentenceTransformer

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
customer_df = pd.read_csv(
    "processed/telco_eda_processed.csv"
)

customer_df["TotalCharges"] = pd.to_numeric(
    customer_df["TotalCharges"],
    errors="coerce"
)

customer_df["TotalCharges"] = customer_df["TotalCharges"].fillna(
    customer_df["TotalCharges"].median()
)

print("Customers:", len(customer_df))

Customers: 7043


In [3]:
def engineer_customer_features(customer_df):

    customer_df = customer_df.copy()

    customer_df["TotalCharges"] = pd.to_numeric(
        customer_df["TotalCharges"],
        errors="coerce"
    )

    customer_df["TotalCharges"] = customer_df["TotalCharges"].fillna(
        customer_df["TotalCharges"].median()
    )

    customer_df["NewCustomer"] = (
        customer_df["tenure"] <= 6
    ).astype(int)

    customer_df["LongTermCustomer"] = (
        customer_df["tenure"] >= 48
    ).astype(int)

    customer_df["ChargeToTenure"] = (
        customer_df["TotalCharges"] /
        (customer_df["tenure"] + 1)
    )

    monthly_threshold = customer_df["MonthlyCharges"].quantile(0.75)

    customer_df["HighMonthlyCharge"] = (
        customer_df["MonthlyCharges"] >= monthly_threshold
    ).astype(int)

    customer_df["HighValueHighRisk"] = (
        (customer_df["MonthlyCharges"] >= monthly_threshold)
        &
        (customer_df["tenure"] <= 6)
    ).astype(int)

    return customer_df

In [4]:
customer_df = pd.read_csv(
    "processed/telco_eda_processed.csv"
)

customer_df = engineer_customer_features(customer_df)

In [5]:
preprocessor = joblib.load(
    "../models/preprocessor.pkl"
)

ml_model = joblib.load(
    "../models/improved_logistic_regression.pkl"
)

print("ML model loaded.")

ML model loaded.


In [6]:
def get_customer_profile(customer_id):

    customer = customer_df[
        customer_df["customerID"] == customer_id
    ]

    if customer.empty:
        return {
            "status": "error",
            "message": "Customer ID not found."
        }

    row = customer.iloc[0]

    return {
        "status": "success",
        "customer_id": row["customerID"],
        "gender": row["gender"],
        "senior_citizen": int(row["SeniorCitizen"]),
        "partner": row["Partner"],
        "dependents": row["Dependents"],
        "tenure_months": int(row["tenure"]),
        "phone_service": row["PhoneService"],
        "internet_service": row["InternetService"],
        "contract": row["Contract"],
        "payment_method": row["PaymentMethod"],
        "monthly_charges": float(row["MonthlyCharges"]),
        "total_charges": float(row["TotalCharges"]),
        "paperless_billing": row["PaperlessBilling"]
    }

In [7]:
test_customer_id = customer_df[
    "customerID"
].iloc[0]

profile = get_customer_profile(
    test_customer_id
)

profile


{'status': 'success',
 'customer_id': '7590-VHVEG',
 'gender': 'Female',
 'senior_citizen': 0,
 'partner': 'Yes',
 'dependents': 'No',
 'tenure_months': 1,
 'phone_service': 'No',
 'internet_service': 'DSL',
 'contract': 'Month-to-month',
 'payment_method': 'Electronic check',
 'monthly_charges': 29.85,
 'total_charges': 29.85,
 'paperless_billing': 'Yes'}

In [8]:
CHURN_THRESHOLD = 0.55


def predict_churn(customer_id):

    customer = customer_df[
        customer_df["customerID"] == customer_id
    ]

    if customer.empty:
        return {
            "status": "error",
            "message": "Customer ID not found."
        }

    X_customer = customer.drop(
        columns=["customerID", "Churn"],
        errors="ignore"
    )

    X_processed = preprocessor.transform(
        X_customer
    )

    probability = ml_model.predict_proba(
        X_processed
    )[0, 1]

    prediction = int(
        probability >= CHURN_THRESHOLD
    )

    risk_level = (
        "High"
        if prediction == 1
        else "Low"
    )

    return {
        "status": "success",
        "customer_id": customer_id,
        "churn_probability": round(
            float(probability),
            4
        ),
        "churn_prediction": prediction,
        "risk_level": risk_level,
        "threshold": CHURN_THRESHOLD
    }

In [9]:
churn_result = predict_churn(
    test_customer_id
)

churn_result

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


{'status': 'success',
 'customer_id': '7590-VHVEG',
 'churn_probability': 0.7568,
 'churn_prediction': 1,
 'risk_level': 'High',
 'threshold': 0.55}

In [10]:
DSC_model = joblib.load(
    "../models/improved_decision_tree.pkl"
)

shap_explainer = shap.TreeExplainer(
    DSC_model
)

print("DscionTree SHAP explainer loaded.")

DscionTree SHAP explainer loaded.


In [11]:
df=customer_df


In [12]:
def get_shap_explanation(customer_id, top_n=5):

    # -----------------------------
    # 1. Get customer
    # -----------------------------
    customer_row = df[df["customerID"] == customer_id].copy()

    if customer_row.empty:
        return {
            "error": f"Customer ID {customer_id} not found."
        }

    # -----------------------------
    # 2. Prepare features
    # -----------------------------
    X_customer = customer_row.drop(
        columns=["customerID", "Churn"],
        errors="ignore"
    )

    # -----------------------------
    # 3. Transform using preprocessor
    # -----------------------------
    X_transformed = preprocessor.transform(X_customer)

    # -----------------------------
    # 4. Get feature names
    # -----------------------------
    feature_names = np.array(
        preprocessor.get_feature_names_out()
    )

    # -----------------------------
    # 5. SHAP values
    # -----------------------------
    shap_values = shap_explainer.shap_values(
        X_transformed
    )

    # -----------------------------
    # 6. Handle SHAP output
    # -----------------------------
    if isinstance(shap_values, list):

        # Binary classification
        if len(shap_values) == 2:
            values = np.asarray(shap_values[1])[0]
        else:
            values = np.asarray(shap_values[0])[0]

    else:

        shap_values = np.asarray(shap_values)

        # Possible shape: (1, features, classes)
        if shap_values.ndim == 3:
            values = shap_values[0, :, 1]

        # Possible shape: (1, features)
        elif shap_values.ndim == 2:
            values = shap_values[0]

        # Possible shape: (features,)
        elif shap_values.ndim == 1:
            values = shap_values

        else:
            raise ValueError(
                f"Unexpected SHAP shape: {shap_values.shape}"
            )

    # -----------------------------
    # 7. Safety check
    # -----------------------------
    if len(values) != len(feature_names):

        raise ValueError(
            f"SHAP values length = {len(values)}, "
            f"but feature names length = {len(feature_names)}"
        )

    # -----------------------------
    # 8. Create SHAP table
    # -----------------------------
    shap_table = pd.DataFrame({
        "feature": feature_names,
        "shap_value": values
    })

    shap_table["absolute_value"] = (
        shap_table["shap_value"].abs()
    )

    # -----------------------------
    # 9. Sort by importance
    # -----------------------------
    shap_table = shap_table.sort_values(
        "absolute_value",
        ascending=False
    ).head(top_n)

    # -----------------------------
    # 10. Positive / negative
    # -----------------------------
    shap_table["direction"] = np.where(
        shap_table["shap_value"] > 0,
        "Increases churn risk",
        "Decreases churn risk"
    )

    return {
        "customer_id": customer_id,
        "top_features": shap_table[
            [
                "feature",
                "shap_value",
                "absolute_value",
                "direction"
            ]
        ].to_dict(orient="records")
    }

In [13]:
shap_result = get_shap_explanation(
    test_customer_id
)

shap_result

{'customer_id': '7590-VHVEG',
 'top_features': [{'feature': 'cat__Contract_Month-to-month',
   'shap_value': 0.2041092166923375,
   'absolute_value': 0.2041092166923375,
   'direction': 'Increases churn risk'},
  {'feature': 'cat__InternetService_Fiber optic',
   'shap_value': -0.1338197698477773,
   'absolute_value': 0.1338197698477773,
   'direction': 'Decreases churn risk'},
  {'feature': 'num__tenure',
   'shap_value': 0.12116269200484933,
   'absolute_value': 0.12116269200484933,
   'direction': 'Increases churn risk'},
  {'feature': 'cat__TechSupport_No',
   'shap_value': 0.06184038917161898,
   'absolute_value': 0.06184038917161898,
   'direction': 'Increases churn risk'},
  {'feature': 'num__TotalCharges',
   'shap_value': 0.02313519949744489,
   'absolute_value': 0.02313519949744489,
   'direction': 'Increases churn risk'}]}

In [14]:
knowledge_index = faiss.read_index(
    "../vector_store/knowledge_faiss.index"
)

with open(
    "../vector_store/knowledge_metadata.pkl",
    "rb"
) as f:
    knowledge_df = pickle.load(f)

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print(
    "Knowledge chunks:",
    knowledge_index.ntotal
)

Knowledge chunks: 6


In [15]:
def search_knowledge_base(
    query,
    top_k=3
):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = knowledge_index.search(
        np.array(query_embedding).astype("float32"),
        top_k
    )

    results = []

    for score, index in zip(
        scores[0],
        indices[0]
    ):

        if index < len(knowledge_df):

            row = knowledge_df.iloc[index]

            results.append({
                "source": row["source"],
                "text": row["text"],
                "similarity_score": round(
                    float(score),
                    4
                )
            })

    return {
        "status": "success",
        "query": query,
        "results": results
    }

In [16]:
knowledge_result = search_knowledge_base(
    "What is the cancellation policy?"
)

knowledge_result

{'status': 'success',
 'query': 'What is the cancellation policy?',
 'results': [{'source': 'cancellation_policy.txt',
   'text': "Cancellation Policy Customers may request cancellation of their service. Customers should review their current contract terms before cancellation. Month-to-month contracts generally provide more flexibility than contracts with longer commitments. Customers considering cancellation may contact support to understand available options and any applicable contractual conditions. The AI assistant must not invent cancellation fees or contract terms. If a specific fee is not present in the knowledge base, the assistant should say that the information is unavailable and recommend checking the customer's contract.",
   'similarity_score': 0.6904},
  {'source': 'billing_policy.txt',
   'text': "Billing Policy Customers are billed according to their selected service plan and subscribed services. Monthly charges may include internet service, phone service, streaming ser

In [17]:
TOOLS = {
    "get_customer_profile": get_customer_profile,
    "predict_churn": predict_churn,
    "get_shap_explanation": get_shap_explanation,
    "search_knowledge_base": search_knowledge_base
}

print("Available tools:")

for tool_name in TOOLS:
    print("✓", tool_name)

Available tools:
✓ get_customer_profile
✓ predict_churn
✓ get_shap_explanation
✓ search_knowledge_base


In [18]:
def call_tool(
    tool_name,
    **kwargs
):

    if tool_name not in TOOLS:
        return {
            "status": "error",
            "message": f"Unknown tool: {tool_name}"
        }

    try:

        result = TOOLS[
            tool_name
        ](**kwargs)

        return result

    except Exception as e:

        return {
            "status": "error",
            "message": str(e)
        }

In [19]:
print(
    call_tool(
        "get_customer_profile",
        customer_id=test_customer_id
    )
)

print()

print(
    call_tool(
        "predict_churn",
        customer_id=test_customer_id
    )
)

print()

print(
    call_tool(
        "get_shap_explanation",
        customer_id=test_customer_id
    )
)

print()

print(
    call_tool(
        "search_knowledge_base",
        query="billing policy"
    )
)

{'status': 'success', 'customer_id': '7590-VHVEG', 'gender': 'Female', 'senior_citizen': 0, 'partner': 'Yes', 'dependents': 'No', 'tenure_months': 1, 'phone_service': 'No', 'internet_service': 'DSL', 'contract': 'Month-to-month', 'payment_method': 'Electronic check', 'monthly_charges': 29.85, 'total_charges': 29.85, 'paperless_billing': 'Yes'}

{'status': 'success', 'customer_id': '7590-VHVEG', 'churn_probability': 0.7568, 'churn_prediction': 1, 'risk_level': 'High', 'threshold': 0.55}

{'customer_id': '7590-VHVEG', 'top_features': [{'feature': 'cat__Contract_Month-to-month', 'shap_value': 0.2041092166923375, 'absolute_value': 0.2041092166923375, 'direction': 'Increases churn risk'}, {'feature': 'cat__InternetService_Fiber optic', 'shap_value': -0.1338197698477773, 'absolute_value': 0.1338197698477773, 'direction': 'Decreases churn risk'}, {'feature': 'num__tenure', 'shap_value': 0.12116269200484933, 'absolute_value': 0.12116269200484933, 'direction': 'Increases churn risk'}, {'feature

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [20]:
def determine_tools(
    question,
    customer_id=None
):

    question_lower = question.lower()

    required_tools = []

    if customer_id is not None:

        required_tools.append(
            "get_customer_profile"
        )

        if any(
            word in question_lower
            for word in [
                "churn",
                "risk",
                "probability",
                "likely to leave"
            ]
        ):
            required_tools.append(
                "predict_churn"
            )

        if any(
            word in question_lower
            for word in [
                "why",
                "reason",
                "factor",
                "contributor",
                "explain"
            ]
        ):
            required_tools.append(
                "get_shap_explanation"
            )

    if any(
        word in question_lower
        for word in [
            "policy",
            "billing",
            "contract",
            "cancel",
            "cancellation",
            "support",
            "procedure",
            "faq",
            "service"
        ]
    ):
        required_tools.append(
            "search_knowledge_base"
        )

    return list(dict.fromkeys(required_tools))

In [21]:
questions = [
    "Why is this customer at high risk?",
    "What is the cancellation policy?",
    "Explain this customer's churn risk and what policy applies."
]

for question in questions:

    selected = determine_tools(
        question,
        customer_id=test_customer_id
    )

    print("\nQUESTION:")
    print(question)

    print("TOOLS:")
    print(selected)


QUESTION:
Why is this customer at high risk?
TOOLS:
['get_customer_profile', 'predict_churn', 'get_shap_explanation']

QUESTION:
What is the cancellation policy?
TOOLS:
['get_customer_profile', 'search_knowledge_base']

QUESTION:
Explain this customer's churn risk and what policy applies.
TOOLS:
['get_customer_profile', 'predict_churn', 'get_shap_explanation', 'search_knowledge_base']


In [22]:
def execute_required_tools(
    customer_id,
    question
):

    selected_tools = determine_tools(
        question,
        customer_id
    )

    outputs = {}

    for tool_name in selected_tools:

        if tool_name == "get_customer_profile":

            outputs[tool_name] = call_tool(
                tool_name,
                customer_id=customer_id
            )

        elif tool_name == "predict_churn":

            outputs[tool_name] = call_tool(
                tool_name,
                customer_id=customer_id
            )

        elif tool_name == "get_shap_explanation":

            outputs[tool_name] = call_tool(
                tool_name,
                customer_id=customer_id
            )

        elif tool_name == "search_knowledge_base":

            outputs[tool_name] = call_tool(
                tool_name,
                query=question
            )

    return {
        "selected_tools": selected_tools,
        "tool_outputs": outputs
    }

In [23]:
question = (
    "Why is this customer at high risk "
    "and what retention policy applies?"
)

workflow_result = execute_required_tools(
    test_customer_id,
    question
)

workflow_result

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


{'selected_tools': ['get_customer_profile',
  'predict_churn',
  'get_shap_explanation',
  'search_knowledge_base'],
 'tool_outputs': {'get_customer_profile': {'status': 'success',
   'customer_id': '7590-VHVEG',
   'gender': 'Female',
   'senior_citizen': 0,
   'partner': 'Yes',
   'dependents': 'No',
   'tenure_months': 1,
   'phone_service': 'No',
   'internet_service': 'DSL',
   'contract': 'Month-to-month',
   'payment_method': 'Electronic check',
   'monthly_charges': 29.85,
   'total_charges': 29.85,
   'paperless_billing': 'Yes'},
  'predict_churn': {'status': 'success',
   'customer_id': '7590-VHVEG',
   'churn_probability': 0.7568,
   'churn_prediction': 1,
   'risk_level': 'High',
   'threshold': 0.55},
  'get_shap_explanation': {'customer_id': '7590-VHVEG',
   'top_features': [{'feature': 'cat__Contract_Month-to-month',
     'shap_value': 0.2041092166923375,
     'absolute_value': 0.2041092166923375,
     'direction': 'Increases churn risk'},
    {'feature': 'cat__InternetS

In [24]:
def build_verified_context(
    workflow_result
):

    context_parts = []

    for tool_name, output in (
        workflow_result["tool_outputs"].items()
    ):

        context_parts.append(
            f"""
TOOL: {tool_name}

RESULT:
{output}
"""
        )

    return "\n".join(
        context_parts
    )

In [25]:
verified_context = build_verified_context(
    workflow_result
)

print(verified_context)


TOOL: get_customer_profile

RESULT:
{'status': 'success', 'customer_id': '7590-VHVEG', 'gender': 'Female', 'senior_citizen': 0, 'partner': 'Yes', 'dependents': 'No', 'tenure_months': 1, 'phone_service': 'No', 'internet_service': 'DSL', 'contract': 'Month-to-month', 'payment_method': 'Electronic check', 'monthly_charges': 29.85, 'total_charges': 29.85, 'paperless_billing': 'Yes'}


TOOL: predict_churn

RESULT:
{'status': 'success', 'customer_id': '7590-VHVEG', 'churn_probability': 0.7568, 'churn_prediction': 1, 'risk_level': 'High', 'threshold': 0.55}


TOOL: get_shap_explanation

RESULT:
{'customer_id': '7590-VHVEG', 'top_features': [{'feature': 'cat__Contract_Month-to-month', 'shap_value': 0.2041092166923375, 'absolute_value': 0.2041092166923375, 'direction': 'Increases churn risk'}, {'feature': 'cat__InternetService_Fiber optic', 'shap_value': -0.1338197698477773, 'absolute_value': 0.1338197698477773, 'direction': 'Decreases churn risk'}, {'feature': 'num__tenure', 'shap_value': 0.1

In [26]:
tool_config = {
    "tools": [
        "get_customer_profile",
        "predict_churn",
        "get_shap_explanation",
        "search_knowledge_base"
    ],
    "workflow_type": "lightweight_local_tool_calling",
    "vector_database": "FAISS",
    "customer_database": "SQLite",
    "ml_model": "Logistic Regression",
    "shap_model": "XGBoost"
}

os.makedirs(
    "../models",
    exist_ok=True
)

with open(
    "../models/tool_config.pkl",
    "wb"
) as f:

    pickle.dump(
        tool_config,
        f
    )

print("Tool configuration saved.")

Tool configuration saved.


In [27]:
demo_questions = [
    "Why is this customer at high risk?",
    "What factors are increasing the customer's churn probability?",
    "What cancellation policy applies to this customer?"
]

for question in demo_questions:

    print("=" * 80)

    print("QUESTION:")
    print(question)

    result = execute_required_tools(
        test_customer_id,
        question
    )

    print("\nTOOLS CALLED:")
    for tool in result["selected_tools"]:
        print("→", tool)

    print("\nVERIFIED OUTPUT:")
    print(
        build_verified_context(result)
    )

QUESTION:
Why is this customer at high risk?

TOOLS CALLED:
→ get_customer_profile
→ predict_churn
→ get_shap_explanation

VERIFIED OUTPUT:

TOOL: get_customer_profile

RESULT:
{'status': 'success', 'customer_id': '7590-VHVEG', 'gender': 'Female', 'senior_citizen': 0, 'partner': 'Yes', 'dependents': 'No', 'tenure_months': 1, 'phone_service': 'No', 'internet_service': 'DSL', 'contract': 'Month-to-month', 'payment_method': 'Electronic check', 'monthly_charges': 29.85, 'total_charges': 29.85, 'paperless_billing': 'Yes'}


TOOL: predict_churn

RESULT:
{'status': 'success', 'customer_id': '7590-VHVEG', 'churn_probability': 0.7568, 'churn_prediction': 1, 'risk_level': 'High', 'threshold': 0.55}


TOOL: get_shap_explanation

RESULT:
{'customer_id': '7590-VHVEG', 'top_features': [{'feature': 'cat__Contract_Month-to-month', 'shap_value': 0.2041092166923375, 'absolute_value': 0.2041092166923375, 'direction': 'Increases churn risk'}, {'feature': 'cat__InternetService_Fiber optic', 'shap_value': -

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
